In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

In [2]:
columns = [
'duration','protocol_type','service','flag','src_bytes',
'dst_bytes','land','wrong_fragment','urgent','hot',
'num_failed_logins','logged_in','num_compromised',
'root_shell','su_attempted','num_root',
'num_file_creations','num_shells','num_access_files',
'num_outbound_cmds','is_host_login','is_guest_login',
'count','srv_count','serror_rate','srv_serror_rate',
'rerror_rate','srv_rerror_rate','same_srv_rate',
'diff_srv_rate','srv_diff_host_rate','dst_host_count',
'dst_host_srv_count','dst_host_same_srv_rate',
'dst_host_diff_srv_rate',
'dst_host_same_src_port_rate',
'dst_host_srv_diff_host_rate',
'dst_host_serror_rate',
'dst_host_srv_serror_rate',
'dst_host_rerror_rate',
'dst_host_srv_rerror_rate',
'label','difficulty'
]

In [3]:
train = pd.read_csv(
    "KDDTrain+.txt",
    names=columns
)

test = pd.read_csv(
    "KDDTest+.txt",
    names=columns
)

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

Train Shape: (125973, 43)
Test Shape: (22544, 43)


In [4]:
train['label'] = train['label'].apply(
    lambda x: 0 if x == 'normal' else 1
)

test['label'] = test['label'].apply(
    lambda x: 0 if x == 'normal' else 1
)

print(train['label'].value_counts())

label
0    67343
1    58630
Name: count, dtype: int64


In [5]:
cat_cols = [
    'protocol_type',
    'service',
    'flag'
]

for col in cat_cols:

    le = LabelEncoder()

    train[col] = le.fit_transform(train[col])

    test[col] = le.transform(test[col])

print("Encoding Complete")

Encoding Complete


In [6]:
X_train = train.drop(
    ['label','difficulty'],
    axis=1
)

y_train = train['label']

X_test = test.drop(
    ['label','difficulty'],
    axis=1
)

y_test = test['label']

print(X_train.shape)
print(X_test.shape)

(125973, 41)
(22544, 41)


In [7]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

print("Training Complete")

Training Complete


In [8]:
preds = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    preds
)

print("Accuracy:", accuracy)

print(classification_report(
    y_test,
    preds
))

Accuracy: 0.7706706884315118
              precision    recall  f1-score   support

           0       0.66      0.97      0.78      9711
           1       0.97      0.62      0.75     12833

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.83      0.77      0.77     22544



In [9]:
print("Train")
print(train['label'].value_counts())

print("\nTest")
print(test['label'].value_counts())

Train
label
0    67343
1    58630
Name: count, dtype: int64

Test
label
1    12833
0     9711
Name: count, dtype: int64


In [10]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, preds)

print(cm)

[[9434  277]
 [4893 7940]]


In [11]:
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.66      0.97      0.78      9711
           1       0.97      0.62      0.75     12833

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.83      0.77      0.77     22544



In [12]:
print(X_train.shape)
print(X_test.shape)

(125973, 41)
(22544, 41)


In [13]:
probs = model.predict_proba(X_test)[:,1]

risk_score = (probs * 100).round(2)

risk_level = []

for score in risk_score:
    if score < 25:
        risk_level.append("Low")
    elif score < 50:
        risk_level.append("Medium")
    elif score < 75:
        risk_level.append("High")
    else:
        risk_level.append("Critical")

results = X_test.copy()

results["Actual"] = y_test
results["Prediction"] = preds
results["Threat_Probability"] = probs
results["Risk_Score"] = risk_score
results["Risk_Level"] = risk_level

results.to_csv("predictions.csv", index=False)

print("predictions.csv created")

predictions.csv created
